# Apresentação

Este é um projeto da disciplina Operacionalização de Modelos com MLOps do Instituto INFNET. Este notebook e os dataset necessários para esse projeto estão disponíveis no repositório: https://github.com/victorlima090/fundamentos-ml-ops.

# Projeto de Disciplina Operacionalização de Modelos com MLOps


# Contexto
No contexto do nosso problema, temos uma empresa que produz e comercializa safras de vinho vermelhas e brancos. Esta empresa gostaria de, antes de mandar uma safra para que receba uma nota de avaliação, realizar uma predição sobre a qualidade do vinho, para que possa decidir enviar somente os vinhos que sabe que terão notas altas, e assim, aumentar a reputação da empresa.

Dessa forma, o dataset a ser analisado consiste de informações químicas de várias vinhos que passaram pela avaliação de especialistas. O resultado final, será uma página, onde é possível imputar as qualidades químicas de um vinho que quer analisar, e receber a probabilidade do mesmo receber uma nota alta, acima de 7, e ser classificado como um vinho recomendado para avaliação humana.

Existem diversas escalas de classificação de vinho utilizadas no mercado, e como o dataset possui uma feature score que vai de 0 a 10, foi escolhido utilizar a escala de Robert Parker, mesma escala utilizada pelo site Wine Spectator, que contem centenas de milhares de avaliações de vinho.
Nessa escala, a nota vai de 0 a 100, e segue as seguintes faixas:
- 96–100 – Extraordinário
- 90–95 – Excepcional
- 80–89 – Pouco acima da média até muito bom
- 70–79 – Médio
- 60–69 – Abaixo da média
- 50–59 – Inaceitável

Dessa forma, para se adaptar ao dataset, a escala foi reduzida para:
- 9,6–10 – Extraordinário
- 9,0–9,5 – Excepcional
- 8,0–8,9 – Pouco acima da média até muito bom
- 7,0–7,9 – Médio
- 6,0–6,9 – Abaixo da média
- 5,0–5,9 – Inaceitável

Foi escolhido como 7 a nota de corte para determinar se um vinho é recomendado ou não.

# Objetivo
O objetivo desse trabalho é criar um projeto de Machine Learning operacional, que contenha as principais etapas do ciclo de vida de um modelo, e que seja facilmente reproduzível através de uma estrutura robusta de pipelines. 

Neste projeto, temos a ingestão de dados, através da API do kaggle, o pré processamento dos mesmo e, por fim, o teste e modelagem. Além disso, foi criada duas páginas web através da ferramenta StreamLit, onde é possível testar o melhor modelo treinado, além de monitorar se o modelo está defasado ou não.


# Fonte de dados
A fonte de dados utilizada é de um trabalho https://www.researchgate.net/publication/221612614_Using_Data_Mining_for_Wine_Quality_Assessment, que coletou amostras do vinho Vinho Verde, produzido na região de Minho, em Portugal. O dataset desse trabalho foi disponibilizado no seguinte link: https://www.kaggle.com/datasets/alirezamehrad/red-and-white-wine-quality. 

# Arquitetura
Este projeto foi separado em três etapas: ingestão, pré processamento e modelagem.


## Ingestão
Esta é a etapa responsável pelo download do dataset e seu armazenamento. Além disso, é possível, através do arquivos de configuração, não realizar o download caso o dataset já tenha sido baixado, tanto quanto é possível uma atualização.

## Pré Processamento
Esta é a etapa que prepara o dataset, realizando diversas transformações para a etapa posterior de modelagem. Essas transformações se mostraram necessárias durante uma análise exploratória prévia do dataset. Segue abaixo as transformações realizadas:

### Transformação Logarítmica
 A transformação logarítmica decorre de algumas features possuírem uma skewness maior que 1.5, logo, possuem uma cauda longa que distorce modelos lineares e aumenta a influência de outliers.
### Razões de Features
Foi verificado que algumas features, quando combinadas entre si, ou seja, trazem maior correlação com a variável target, se forem unidas através de uma razão, como "ph/alcohol". Dessa forma, novas features baseadas nas originais foram criadas para o dataset.

### Seleção de Features
Nesta etapa final, retira-se algumas das colunas originais, pois o dataset cresceu em dimensionalidade, o que aumenta o custo operacional.

## Modelagem

Nesta etapa, o dataset já preparado, será separado nos conjuntos de treino e teste para que não haja vazamento de dados. Em seguida, de acordo com o arquivo de configuração, será realizado um teste com diversos modelos diferentes, em que será feita uma otimização dos seus parâmetros com a ferramenta Optuna. Os dados dos treinamentos desses modelos serão armazenados num servidor do MlFlow, onde o melhor modelo, através da métrica escolhida, será armazenado para que possa ser utilizada em um ambiente produtivo.

Neste trabalho, foi decidido que a métrica principal a ser otimizada no modelo é a precisão, uma vez que, é preferível evitar ao máximo que vinhos de baixa qualidade sejam mandados para a avaliação. Além disso, pelo dataset ser desbalanceado, no caso ontem temos uma maioria de vinhos de qualidade abaixo de 7, a precisão se torna uma métrica melhor do que a acurácia.

## Modelos Escolhidos
### LogisticRegression
#### Vantagens
Altamente interpretável — os coeficientes indicam diretamente o peso de cada feature
Treinamento muito rápido e escalável
Probabilidades bem calibradas nativamente (sem pós-processamento)
Funciona bem combinado com PCA/RFE upstream
#### Desvantagens
Assume relação linear entre features e log-odds — incapaz de capturar não-linearidades
Péssimo com features altamente correlacionadas sem regularização
Sensível a outliers quando os dados não estão escalonados
### DecisionTreeClassifier
#### Vantagens
Totalmente interpretável — pode ser visualizado como diagrama de decisão
Não requer escalonamento de features
Lida com classes desbalanceadas via class_weight: balanced
Captura interações não-lineares naturalmente
#### Desvantagens
Alto risco de overfitting sem controle de max_depth
Instável — pequenas mudanças nos dados podem alterar toda a estrutura da árvore
Probabilidades pouco calibradas (calculadas por proporção em cada folha)
### KNeighborsClassifier
#### Vantagens
Sem fase de treino (lazy learner) — simples de implementar
Captura fronteiras de decisão complexas e irregulares
Naturalmente multi-classe
#### Desvantagens
Inferência lenta em datasets grandes — O(n) por predição
Sofre com a maldição da dimensionalidade em espaços de alta dimensão
Não possui class_weight — problemático com classes desbalanceadas (~10% positivos no seu caso)
### SVC
#### Vantagens
Eficaz em espaços de alta dimensionalidade
Kernel RBF captura fronteiras de decisão não-lineares complexas
Robusto a outliers pela natureza da margem de separação
#### Desvantagens
Treino O(n²) — lento para datasets grandes (mitigado no YAML com max_samples_for_tuning: 5000)
Requer probability: true para retornar probabilidades via Platt scaling — não está configurado no seu YAML, o que vai causar erro em produção
Difícil de interpretar
### RandomForestClassifier
#### Vantagens
Robusto a overfitting pela média entre muitas árvores descorrelacionadas
Feature importance nativa e confiável
Pouco sensível a outliers e ruído nos dados
#### Desvantagens
Treinamento mais lento que uma árvore única
Probabilidades moderadamente calibradas — pode precisar de CalibratedClassifierCV
class_weight não está configurado no YAML — problemático com classes desbalanceadas
### GradientBoostingClassifier
#### Vantagens
Geralmente melhor desempenho preditivo entre modelos baseados em árvores
Captura padrões complexos pelo treinamento sequencial corrigindo erros
predict_proba nativo e razoavelmente calibrado
#### Desvantagens
Treinamento sequencial — não paralelizável como Random Forest
Sensível ao learning_rate: valores altos causam overfitting, baixos requerem mais estimadores
Nenhum mecanismo de class_weight configurado no YAML
### XGBClassifier
#### Vantagens
Regularização L1 (reg_alpha) e L2 nativas — reduz overfitting
scale_pos_weight permite lidar explicitamente com classes desbalanceadas
colsample_bytree reduz correlação entre árvores, aumentando diversidade do ensemble
#### Desvantagens
Maior número de hiperparâmetros para ajustar
scale_pos_weight não está configurado no YAML — recomendado definir como ~9 (ratio negativo/positivo) para o seu problema
Menos interpretável que modelos lineares



## Técnicas de Redução de Dimensionalidade

## Métricas analisadas 

## Modelo Final Escolhido